In [3]:
import cv2
import os

video_path = 'data/raw_video/WIN_20260702_15_01_44_Pro.mp4'  # Tên file video bạn vừa quay
output_dir = 'data/raw_video/images'       # Thư mục lưu ảnh cắt ra
os.makedirs(output_dir, exist_ok=True)

cap = cv2.VideoCapture(video_path)
count = 0
frame_rate = 5 # Cứ cách 5 frames lấy 1 ảnh để tránh ảnh bị trùng lặp quá nhiều

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    if count % frame_rate == 0:
        # Lưu ảnh vào thư mục
        img_name = os.path.join(output_dir, f"frame_1_{count:04d}.jpg")
        cv2.imwrite(img_name, frame)
        
    count += 1

cap.release()
print(f"Đã cắt xong! Toàn bộ ảnh thô nằm trong thư mục: '{output_dir}'")


Đã cắt xong! Toàn bộ ảnh thô nằm trong thư mục: 'data/raw_video/images'


In [8]:
# dùng model để nhận diện tay và vẽ bounding box quanh tay và lưu ảnh đã vẽ bounding box vào thư mục khác và lưu vị trí bounding box vào file txt
from ultralytics import YOLO
model = YOLO('models/best_model_m1.pt')  # Load model YOLOv8n

# Chạy predict trên cả thư mục ảnh thô
model.predict(
    source='data/raw_video/images',  # Thư mục chứa ảnh thô
    save=True,          # Lưu ảnh đã vẽ khung để bạn xem thử
    save_txt=True,      # CỰC KỲ QUAN TRỌNG: Tự động tạo file nhãn .txt chuẩn YOLO
    conf=0.25           # Độ tự tin tối thiểu để nhận diện (chỉnh thấp xuống nếu muốn lấy nhiều nhãn)
)


image 1/182 d:\vscode\Python\projects\sign-language-pipline\data\raw_video\images\frame_1_0000.jpg: 384x640 1 hand, 26.6ms
image 2/182 d:\vscode\Python\projects\sign-language-pipline\data\raw_video\images\frame_1_0005.jpg: 384x640 1 hand, 43.0ms
image 3/182 d:\vscode\Python\projects\sign-language-pipline\data\raw_video\images\frame_1_0010.jpg: 384x640 1 hand, 2.9ms
image 4/182 d:\vscode\Python\projects\sign-language-pipline\data\raw_video\images\frame_1_0015.jpg: 384x640 1 hand, 2.9ms
image 5/182 d:\vscode\Python\projects\sign-language-pipline\data\raw_video\images\frame_1_0020.jpg: 384x640 (no detections), 2.9ms
image 6/182 d:\vscode\Python\projects\sign-language-pipline\data\raw_video\images\frame_1_0025.jpg: 384x640 1 hand, 3.1ms
image 7/182 d:\vscode\Python\projects\sign-language-pipline\data\raw_video\images\frame_1_0030.jpg: 384x640 (no detections), 3.0ms
image 8/182 d:\vscode\Python\projects\sign-language-pipline\data\raw_video\images\frame_1_0035.jpg: 384x640 1 hand, 3.0ms
ima

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'hand'}
 obb: None
 orig_img: array([[[128, 155, 175],
         [129, 156, 176],
         [131, 158, 179],
         ...,
         [169, 189, 214],
         [171, 188, 214],
         [171, 188, 214]],
 
        [[129, 156, 176],
         [130, 157, 177],
         [132, 159, 180],
         ...,
         [167, 187, 212],
         [169, 186, 212],
         [170, 187, 213]],
 
        [[132, 156, 178],
         [133, 157, 179],
         [135, 159, 183],
         ...,
         [166, 185, 212],
         [168, 187, 214],
         [170, 189, 216]],
 
        ...,
 
        [[112, 142, 161],
         [108, 138, 157],
         [103, 133, 152],
         ...,
         [103, 132, 163],
         [103, 131, 162],
         [101, 130, 161]],
 
        [[101, 133, 152],
         [100, 132, 151],
         [102, 132, 151],
         ...,
         [102, 133, 

In [ ]:
import os

# Thay đổi đường dẫn đến 2 thư mục của bạn
image_dir = 'runs/detect/predict-6'
label_dir = 'runs/detect/predict-6/labels'
# image_dir = 'data/processed_frames/hand_train_fine_tune/images'
# label_dir = 'data/processed_frames/hand_train_fine_tune/labels'

# Lấy danh sách tên file (không bao gồm đuôi .jpg hay .txt)
images = {os.path.splitext(f)[0] for f in os.listdir(image_dir) if f.endswith(('.jpg', '.jpeg', '.png'))}
labels = {os.path.splitext(f)[0] for f in os.listdir(label_dir) if f.endswith('.txt')}

# Tìm các file .txt có nhãn nhưng ảnh đã bị bạn xóa mất
orphan_labels = labels - images

# Tiến hành xóa các file .txt thừa
for name in orphan_labels:
    txt_path = os.path.join(label_dir, name + '.txt')
    if os.path.exists(txt_path):
        os.remove(txt_path)
        print(f"Đã xóa file nhãn thừa: {txt_path}")

print("=== ĐÃ ĐỒNG BỘ XONG! Thư mục nhãn đã sạch sẽ ===")

# tìm các ảnh không có file nhãn tương ứng và tạo file nhãn rỗng cho chúng
orphan_images = images - labels

for name in orphan_images:
    txt_path = os.path.join(label_dir, name + '.txt')
    if not os.path.exists(txt_path):
        with open(txt_path, 'w') as f:
            pass  # Tạo file nhãn rỗng
        print(f"Đã tạo file nhãn rỗng: {txt_path}")

# đếm số lượng ảnh và số lượng file nhãn trong thư mục
image_count = len([f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.jpeg', '.png'))])
label_count = len([f for f in os.listdir(label_dir) if f.endswith('.txt')])

print(f"Số lượng ảnh trong thư mục: {image_count}")
print(f"Số lượng file nhãn trong thư mục: {label_count}")

=== ĐÃ ĐỒNG BỘ XONG! Thư mục nhãn đã sạch sẽ ===
Số lượng ảnh trong thư mục: 154
Số lượng file nhãn trong thư mục: 154


In [4]:
#xóa cái ảnh ko có file nhãn tương ứng

images_dir = 'data/processed_frames/hand_train_fine_tune/images'
labels_dir = 'data/processed_frames/hand_train_fine_tune/labels'

# Lấy danh sách các file ảnh
image_files = [f for f in os.listdir(images_dir) if f.endswith('.jpg')]

# Lấy danh sách các file nhãn
label_files = [f for f in os.listdir(labels_dir) if f.endswith('.txt')]


# đếm số lượng ảnh và số lượng file nhãn trong thư mục trước khi xóa
image_count = len([f for f in os.listdir(images_dir) if f.endswith('.jpg')])
label_count = len([f for f in os.listdir(labels_dir) if f.endswith('.txt')])

print(f"Số lượng ảnh trong thư mục trước khi xóa: {image_count}")
print(f"Số lượng file nhãn trong thư mục trước khi xóa: {label_count}")

# Xóa các file ảnh không có file nhãn tương ứng
for image_file in image_files:
    label_file = image_file.replace('.jpg', '.txt')
    if label_file not in label_files:
        os.remove(os.path.join(images_dir, image_file))
        print(f"Đã xóa file ảnh không có nhãn tương ứng: {image_file}")
        
# đếm số lượng ảnh và số lượng file nhãn trong thư mục sau khi xóa
image_count = len([f for f in os.listdir(images_dir) if f.endswith('.jpg')])
label_count = len([f for f in os.listdir(labels_dir) if f.endswith('.txt')])

print(f"Số lượng ảnh trong thư mục sau khi xóa: {image_count}")
print(f"Số lượng file nhãn trong thư mục sau khi xóa: {label_count}")

Số lượng ảnh trong thư mục trước khi xóa: 273
Số lượng file nhãn trong thư mục trước khi xóa: 154
Đã xóa file ảnh không có nhãn tương ứng: frame_0000.jpg
Đã xóa file ảnh không có nhãn tương ứng: frame_0005.jpg
Đã xóa file ảnh không có nhãn tương ứng: frame_0010.jpg
Đã xóa file ảnh không có nhãn tương ứng: frame_0015.jpg
Đã xóa file ảnh không có nhãn tương ứng: frame_0040.jpg
Đã xóa file ảnh không có nhãn tương ứng: frame_0055.jpg
Đã xóa file ảnh không có nhãn tương ứng: frame_0060.jpg
Đã xóa file ảnh không có nhãn tương ứng: frame_0100.jpg
Đã xóa file ảnh không có nhãn tương ứng: frame_0105.jpg
Đã xóa file ảnh không có nhãn tương ứng: frame_0110.jpg
Đã xóa file ảnh không có nhãn tương ứng: frame_0115.jpg
Đã xóa file ảnh không có nhãn tương ứng: frame_0130.jpg
Đã xóa file ảnh không có nhãn tương ứng: frame_0135.jpg
Đã xóa file ảnh không có nhãn tương ứng: frame_0140.jpg
Đã xóa file ảnh không có nhãn tương ứng: frame_0145.jpg
Đã xóa file ảnh không có nhãn tương ứng: frame_0150.jpg
Đã xóa